# Continuidade 03 · Análise

## Contexto

Este notebook responde a quatro perguntas sobre a continuidade do fornecimento nas 33 distribuidoras de grande porte, entre 2022 e 2025, a partir das tabelas da Gold construídas em `continuity/02_gold_continuity`.

Dois indicadores, ambos por consumidor e por ano:

- **DEC-FI:** duração das interrupções, em horas. Quanto tempo, em média, cada consumidor ficou sem energia.
- **FEC-FI:** frequência das interrupções. Quantas vezes, em média, cada consumidor ficou sem energia.

O sufixo FI (falha interna) indica que os indicadores contam as interrupções não programadas originadas no sistema da própria distribuidora, incluindo as ocorridas em dias críticos e em situações de emergência. Ficam de fora apenas as interrupções programadas e as de origem externa. Os indicadores normativos da ANEEL (DEC e FEC) fazem o recorte oposto nesses dois pontos: excluem dias críticos e emergências e incluem as programadas. O DEC-FI mede, assim, a falta de energia que o consumidor sente e que a distribuidora pode gerir.

"Brasil", nos gráficos, é o indicador do conjunto das 33 distribuidoras: a média de cada uma ponderada pelo seu número de consumidores, a mesma regra do PRODIST para agregar conjuntos.

| Pergunta | O que responde |
|---|---|
| O consumidor ficou menos tempo sem energia? | Evolução do conjunto, com o indicador normativo como referência |
| Quem melhorou, partindo de onde? | Ponto de partida, ponto de chegada e trajetória de cada distribuidora |
| Menos interrupções ou interrupções mais curtas? | Duração e frequência lado a lado |
| A regra de agregação do PRODIST muda o resultado? | Diferença entre a regra e a soma simples dos meses |


#### Por que os números de continuidade diferem entre as duas análises

A análise de continuidade (`continuity/03_analysis`) compara anos civis completos, 2022 contra 2025, porque é assim que o PRODIST (Procedimentos de Distribuição da ANEEL) apura o indicador anual. A análise de reclamações (`complaints/04_analysis`) compara janelas de 12 meses encerradas em dez/2024 e jun/2026, para medir a continuidade no mesmo período das reclamações. Os dois usam o mesmo DEC-FI e o mesmo FEC-FI; o período diferente leva a variações diferentes.

O conjunto também muda em uma distribuidora: a continuidade analisa as 33 de grande porte, e as reclamações, 32, porque a CELESC está fora dos rankings de reclamações por qualidade insuficiente do dado.

## Configuração

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.patches import Rectangle
from matplotlib.transforms import Bbox
from pyspark.sql import functions as F
from scipy.stats import spearmanr

# Walk up from the working directory until the folder holding `src` is found,
# so the notebook works at any depth inside notebooks/
REPO_ROOT = os.getcwd()
while not os.path.isdir(os.path.join(REPO_ROOT, "src")):
    parent = os.path.dirname(REPO_ROOT)
    if parent == REPO_ROOT:
        raise FileNotFoundError("Repository root with a src folder not found above " + os.getcwd())
    REPO_ROOT = parent
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from src.config import CATALOG, SCHEMA_GOLD

GOLD = f"{CATALOG}.{SCHEMA_GOLD}"

# Same palette and chart style as complaints/04_analysis
COR_MELHORA, COR_PIORA = "#0C8F8F", "#FFAD08"
COR_NEUTRA, COR_TEXTO, COR_EIXO = "#b4b2ab", "#52514e", "#8a8983"
COR_ALERTA, COR_PONTO, COR_NOME, COR_BRASIL = "#c62828", "#8a8983", "#9e9d98", "#2b2b29"
FUNDO_BOM, FUNDO_PIORA, FUNDO_NEUTRO = "#e3f1f1", "#fff4dc", "#f4f3ef"

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelsize": 9, "axes.labelcolor": COR_TEXTO, "axes.edgecolor": "#c8c7c1",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False,
    "xtick.color": COR_TEXTO, "ytick.color": COR_TEXTO, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.frameon": False, "legend.fontsize": 8, "axes.titlepad": 12,
})


def br(v, casas=2):
    """Number with Portuguese decimal comma."""
    return f"{v:.{casas}f}".replace(".", ",")


def grade_leve(ax, eixo="x"):
    """Very light reference grid on a single axis."""
    ax.grid(True, axis=eixo, color="#ecebe7", linewidth=0.8)
    ax.set_axisbelow(True)


def faixa_eixo(valores):
    """Axis range covering every point except the far ones (boxplot rule), always including zero."""
    v = np.asarray(valores, dtype=float)
    q1, q3 = np.percentile(v, [25, 75])
    iqr = q3 - q1
    dentro = v[(v >= q1 - 1.5 * iqr) & (v <= q3 + 1.5 * iqr)]
    lo, hi = min(dentro.min(), 0.0), max(dentro.max(), 0.0)
    folga = (hi - lo) * 0.08
    return lo - folga, hi + folga


def rotular_pontos(ax, xs, ys, textos, destaque, cores):
    """Name every point without overlaps: highlighted names in the point color, the others small and gray."""
    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    area = ax.get_window_extent(renderer)
    pixels = ax.transData.transform(np.column_stack([xs.values, ys.values]))
    pontos = {dx: Bbox.from_bounds(px - 4, py - 4, 8, 8) for dx, (px, py) in zip(xs.index, pixels)}
    ocupado = []
    candidatos = [(6, 0, "left", "center"), (-6, 0, "right", "center"), (0, 6, "center", "bottom"),
                  (0, -6, "center", "top"), (5, 5, "left", "bottom"), (-5, 5, "right", "bottom"),
                  (5, -5, "left", "top"), (-5, -5, "right", "top"), (12, 0, "left", "center"),
                  (-12, 0, "right", "center"), (0, 13, "center", "bottom"), (0, -13, "center", "top")]
    ordem = [dx for dx in xs.index if dx in destaque] + [dx for dx in xs.index if dx not in destaque]
    for dx in ordem:
        estilo = (dict(fontsize=7.5, color=cores[dx]) if dx in destaque
                  else dict(fontsize=6.5, color=COR_NOME))
        outros = [b for k, b in pontos.items() if k != dx]
        escolhido = None
        for ox, oy, ha, va in candidatos:
            t = ax.annotate(textos[dx], (xs[dx], ys[dx]), xytext=(ox, oy), textcoords="offset points",
                            ha=ha, va=va, zorder=4, annotation_clip=False, **estilo)
            caixa = t.get_window_extent(renderer)
            livre = not any(caixa.overlaps(b) for b in ocupado + outros)
            dentro = area.x0 <= caixa.x0 and caixa.x1 <= area.x1 and area.y0 <= caixa.y0 and caixa.y1 <= area.y1
            if livre and dentro:
                escolhido = caixa
                break
            t.remove()
        if escolhido is None:
            # No free spot: place the label toward the inside of the chart
            direita = pixels[list(xs.index).index(dx)][0] > (area.x0 + area.x1) / 2
            t = ax.annotate(textos[dx], (xs[dx], ys[dx]), xytext=(-6 if direita else 6, 0),
                            textcoords="offset points", ha="right" if direita else "left", va="center",
                            zorder=4, annotation_clip=False, **estilo)
            escolhido = t.get_window_extent(renderer)
        ocupado.append(escolhido)


def matriz_quadrantes(ax, x, y, cores, destaque, fundos):
    """Four-quadrant scatter cut at zero, with every company named and far points pinned to the border."""
    (x0, x1), (y0, y1) = faixa_eixo(x), faixa_eixo(y)
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    for (xp, yp), cor_fundo in fundos.items():
        ax.add_patch(Rectangle((0 if xp else x0, 0 if yp else y0),
                               (x1 if xp else 0) - (0 if xp else x0),
                               (y1 if yp else 0) - (0 if yp else y0),
                               facecolor=cor_fundo, edgecolor="none", zorder=0))
    ax.axvline(0, color=COR_EIXO, linewidth=0.8, zorder=1)
    ax.axhline(0, color=COR_EIXO, linewidth=0.8, zorder=1)
    xs, ys = x.clip(x0, x1), y.clip(y0, y1)
    fora = (xs != x) | (ys != y)
    ax.scatter(xs[~fora], ys[~fora], c=cores[~fora].tolist(), s=38, edgecolor="white", linewidth=0.8, zorder=3)
    ax.scatter(xs[fora], ys[fora], c=cores[fora].tolist(), s=60, marker="D", edgecolor="white", linewidth=0.8,
               zorder=3, clip_on=False)
    textos = {dx: (f"{dx} ({x[dx]:+.0f}%; {y[dx]:+.0f}%)" if fora[dx] else dx) for dx in x.index}
    rotular_pontos(ax, xs, ys, textos, set(destaque) | set(fora[fora].index), cores)


def esquema_quadrantes(titulo, eixo_x, eixo_y, textos, fundos):
    """Explanatory 2x2 diagram: one caption per quadrant, keyed by (x >= 0, y >= 0)."""
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    for (xp, yp), texto in textos.items():
        x0, y0 = (0.5 if xp else 0.0), (0.5 if yp else 0.0)
        ax.add_patch(Rectangle((x0, y0), 0.5, 0.5, facecolor=fundos[(xp, yp)],
                               edgecolor="white", linewidth=4))
        ax.text(x0 + 0.25, y0 + 0.25, texto, ha="center", va="center", fontsize=9.5, color="#2b2b29")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xticks([0.25, 0.75], ["em queda", "em alta"])
    ax.set_yticks([0.25, 0.75], ["em queda", "em alta"])
    ax.tick_params(length=0, labelsize=9)
    for lado in ax.spines.values():
        lado.set_visible(False)
    ax.set_xlabel(eixo_x, fontsize=10)
    ax.set_ylabel(eixo_y, fontsize=10)
    ax.set_title(titulo)
    plt.tight_layout()
    plt.show()


anual = (spark.table(f"{GOLD}.fato_continuidade_anual")
         .withColumn("sig_agente", F.trim("sig_agente")).toPandas())
ranking = (spark.table(f"{GOLD}.ranking_evolucao")
           .withColumn("sig_agente", F.trim("sig_agente")).toPandas()
           .set_index("sig_agente").sort_values("posicao"))

ANOS = sorted(anual["ano"].unique())
ANO_INI, ANO_FIM = ANOS[0], ANOS[-1]
N_DX = anual["num_cnpj"].nunique()
NOME_BRASIL = f"Brasil ({N_DX} distribuidoras)"


def brasil(ano, indicador):
    """Indicator of the whole set in one year: each company weighted by its consumer units."""
    t = anual[anual["ano"] == ano]
    return (t[indicador] * t["nuc_gk"]).sum() / t["nuc_gk"].sum()


def var_pct(ini, fim):
    """Percent change between two values."""
    return (fim / ini - 1) * 100


print(f"Anos............: {ANOS[0]} a {ANOS[-1]}")
print(f"Distribuidoras..: {N_DX}")
print(f"Linhas anuais...: {len(anual)}")

## O consumidor ficou menos tempo sem energia?

A primeira pergunta olha o conjunto das distribuidoras. A linha cheia é o indicador deste trabalho; a tracejada, o indicador normativo da ANEEL, como referência. As duas medem a mesma rede, com recortes diferentes: o DEC-FI conta os dias críticos e as emergências e deixa de fora as interrupções programadas; o normativo faz o contrário.

#### Como ler

À esquerda, a duração (horas sem energia por consumidor no ano); à direita, a frequência (vezes sem energia por consumidor no ano). O título traz a variação do indicador deste trabalho entre o primeiro e o último ano.

In [ ]:
serie = pd.DataFrame({c: [brasil(a, c) for a in ANOS] for c in ["dec_fi", "dec", "fec_fi", "fec"]}, index=ANOS)

fig, eixos = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (fi, norma, nome, unidade) in zip(eixos, [("dec_fi", "dec", "DEC", "horas sem energia"),
                                                  ("fec_fi", "fec", "FEC", "vezes sem energia")]):
    ax.plot(ANOS, serie[fi], color=COR_MELHORA, marker="o", linewidth=2.2, label=f"{nome}-FI (este trabalho)")
    ax.plot(ANOS, serie[norma], color=COR_NEUTRA, marker="o", linewidth=1.6, linestyle="--",
            label=f"{nome} normativo (ANEEL)")
    for a in ANOS:
        ax.annotate(br(serie.loc[a, fi]), (a, serie.loc[a, fi]), xytext=(0, 7), textcoords="offset points",
                    ha="center", fontsize=8, color=COR_MELHORA, fontweight="bold")
        ax.annotate(br(serie.loc[a, norma]), (a, serie.loc[a, norma]), xytext=(0, -12),
                    textcoords="offset points", ha="center", fontsize=7.5, color=COR_TEXTO)
    ax.set_xticks(ANOS)
    ax.set_ylim(0, serie[[fi, norma]].max().max() * 1.25)
    grade_leve(ax, "y")
    ax.set_ylabel(f"{unidade}, por consumidor no ano")
    v = var_pct(serie.loc[ANO_INI, fi], serie.loc[ANO_FIM, fi])
    ax.set_title(f"{nome}-FI do Brasil: {'+' if v >= 0 else ''}{br(v, 1)}% de {ANO_INI} a {ANO_FIM}")
    ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

for c in serie.columns:
    print(f"{c:<7}: {ANO_INI} {serie.loc[ANO_INI, c]:6.2f} | pico {serie[c].max():6.2f} em {serie[c].idxmax()} "
          f"| {ANO_FIM} {serie.loc[ANO_FIM, c]:6.2f} | variacao {var_pct(serie.loc[ANO_INI, c], serie.loc[ANO_FIM, c]):+.1f}%")

**Conclusão:**

No conjunto das 33 distribuidoras, o DEC-FI passou de 15,07 horas em 2022 para 15,88 em 2025 (+5,4%), com pico de 19,33 horas em 2024. O FEC-FI ficou praticamente estável, de 6,52 para 6,59 interrupções (+1,1%). No mesmo período, o DEC normativo caiu 16,6%, e o FEC normativo, 14,6%.

Pela régua regulatória, a continuidade melhorou. Pela régua do consumidor, que sente também a falta de energia em dias críticos e emergências, não melhorou.

## Quem melhorou, partindo de onde?

O ranking de continuidade ordena as distribuidoras pela variação percentual do DEC-FI entre 2022 e 2025. O percentual sozinho engana: cair de 30 para 20 horas e de 6 para 4 horas é a mesma redução de 33%, mas o consumidor da primeira ganhou 10 horas e o da segunda, 2. O gráfico mostra as duas coisas.

#### Como ler

Cada linha é uma distribuidora, na ordem do ranking (a primeira teve a maior redução percentual). O ponto cinza é o DEC-FI de 2022; a ponta da seta, o de 2025. Seta azul para a esquerda: menos horas sem energia. Seta laranja para a direita: mais horas. À direita, a variação percentual. A linha escura é o Brasil.

In [ ]:
tab = ranking[["dec_fi_inicio", "dec_fi_fim", "dec_fi_var_pct"]].copy()
tab.loc[NOME_BRASIL] = [serie.loc[ANO_INI, "dec_fi"], serie.loc[ANO_FIM, "dec_fi"],
                        var_pct(serie.loc[ANO_INI, "dec_fi"], serie.loc[ANO_FIM, "dec_fi"])]
tab = tab.sort_values("dec_fi_var_pct")

fig, ax = plt.subplots(figsize=(10, len(tab) * 0.27 + 1.2))
posicoes_y = np.arange(len(tab))[::-1]
for y, (dx, r) in zip(posicoes_y, tab.iterrows()):
    cor = COR_BRASIL if dx == NOME_BRASIL else (COR_MELHORA if r["dec_fi_fim"] < r["dec_fi_inicio"] else COR_PIORA)
    ax.annotate("", xy=(r["dec_fi_fim"], y), xytext=(r["dec_fi_inicio"], y),
                arrowprops=dict(arrowstyle="-|>", color=cor, linewidth=1.8, shrinkA=0, shrinkB=0,
                                mutation_scale=9))
    ax.scatter(r["dec_fi_inicio"], y, s=20, color=COR_NEUTRA, zorder=3)
    ax.text(1.01, y, f"{r['dec_fi_var_pct']:+.0f}%", transform=ax.get_yaxis_transform(),
            va="center", fontsize=7.5, color=cor, fontweight="bold" if dx == NOME_BRASIL else "normal")
ax.set_yticks(posicoes_y, tab.index)
for rotulo_y in ax.get_yticklabels():
    if rotulo_y.get_text() == NOME_BRASIL:
        rotulo_y.set_fontweight("bold")
ax.tick_params(axis="y", length=0)
ax.set_xlim(0, tab[["dec_fi_inicio", "dec_fi_fim"]].max().max() * 1.05)
grade_leve(ax, "x")
ax.set_xlabel(f"DEC-FI (horas sem energia por consumidor no ano): ponto em {ANO_INI}, seta até {ANO_FIM}")
ax.set_title(f"DEC-FI de {ANO_INI} a {ANO_FIM}, na ordem do ranking de continuidade")
plt.tight_layout()
plt.show()

rho, p = spearmanr(ranking["dec_fi_inicio"], ranking["dec_fi_var_pct"])
print(f"reduziram o DEC-FI...........: {(ranking['dec_fi_var_abs'] < 0).sum()} de {len(ranking)}")
print(f"maiores reducoes em horas....: {ranking['dec_fi_var_abs'].nsmallest(3).round(2).to_dict()}")
print(f"maiores aumentos em horas....: {ranking['dec_fi_var_abs'].nlargest(3).round(2).to_dict()}")
print(f"ponto de partida x variacao..: Spearman {rho:.2f}, valor-p {p:.3f}")

**Conclusão:**

15 das 33 distribuidoras reduziram o DEC-FI. As maiores reduções em horas vieram de quem partia de patamar alto: EQUATORIAL AL (−11,45 h), EQUATORIAL GO (−11,02 h) e EQUATORIAL MA (−10,58 h), todas acima de 28 horas em 2022. Quanto mais alto o ponto de partida, maior tende a ser a redução percentual: o alinhamento entre as duas ordenações, medido pelo Spearman (de −1 a +1), é de −0,40, com valor-p (chance de o alinhamento aparecer por acaso) de 0,022.

No outro extremo, a ELETROPAULO mais que dobrou o DEC-FI, de 8,80 para 20,14 horas (+129%), e a CEEE-D teve o segundo maior aumento em horas (+9,64 h).

#### A melhora foi contínua?

A comparação entre 2022 e 2025 não diz o que aconteceu no meio. Cada célula traz o DEC-FI do ano, em horas; a cor compara com 2022 da mesma distribuidora: azul, abaixo; laranja, acima; quanto mais forte, maior a diferença. As linhas seguem a ordem do ranking.

In [ ]:
horas = anual.pivot(index="sig_agente", columns="ano", values="dec_fi").loc[ranking.index]
horas.loc[NOME_BRASIL] = serie["dec_fi"]
horas = horas.loc[[NOME_BRASIL] + list(ranking.index)]
indice = horas.div(horas[ANO_INI], axis=0) * 100

mapa = LinearSegmentedColormap.from_list("continuidade", [COR_MELHORA, "#ffffff", COR_PIORA])
escala = TwoSlopeNorm(vmin=40, vcenter=100, vmax=250)

fig, ax = plt.subplots(figsize=(7, len(horas) * 0.27 + 1.2))
ax.imshow(indice.clip(40, 250).values, cmap=mapa, norm=escala, aspect="auto")
for i, dx in enumerate(horas.index):
    for j, a in enumerate(ANOS):
        forte = abs(indice.loc[dx, a] - 100) > 60
        ax.text(j, i, br(horas.loc[dx, a], 1), ha="center", va="center", fontsize=7,
                color="white" if forte and indice.loc[dx, a] < 100 else "#2b2b29",
                fontweight="bold" if dx == NOME_BRASIL else "normal")
ax.set_xticks(range(len(ANOS)), ANOS)
ax.set_yticks(range(len(horas)), horas.index)
for rotulo_y in ax.get_yticklabels():
    if rotulo_y.get_text() == NOME_BRASIL:
        rotulo_y.set_fontweight("bold")
ax.xaxis.tick_top()
ax.tick_params(length=0)
for lado in ax.spines.values():
    lado.set_visible(False)
ax.set_title(f"DEC-FI por ano, em horas (cor: comparação com {ANO_INI})", pad=24)
plt.tight_layout()
plt.show()

meio = [a for a in ANOS if a not in (ANO_INI, ANO_FIM)]
pior_no_meio = (horas.loc[ranking.index, meio].max(axis=1) > horas.loc[ranking.index, [ANO_INI, ANO_FIM]].max(axis=1))
print(f"pior ano entre {meio[0]} e {meio[-1]}: {pior_no_meio.sum()} de {len(ranking)} distribuidoras")

**Conclusão:**

Em 24 das 33 distribuidoras, o pior ano foi 2023 ou 2024, e não uma das pontas da comparação. A CEEE-D chegou a 109,19 horas em 2024 e voltou a 40,30 em 2025; a RGE SUL foi a 37,45 horas em 2024 e terminou em 10,95. O ranking entre 2022 e 2025 deve ser lido junto com essa trajetória: com outros anos nas pontas, as posições mudariam.

## Menos interrupções ou interrupções mais curtas?

O DEC-FI mede quanto tempo o consumidor ficou sem energia; o FEC-FI, quantas vezes. Uma distribuidora pode reduzir um e não o outro: restabelecer mais rápido encurta as interrupções sem torná-las mais raras, e manutenção preventiva evita interrupções sem mudar o tempo de reparo. O esquema abaixo mostra o que cada quadrante significa.

In [ ]:
FUNDO_Q = {(False, False): FUNDO_BOM, (False, True): FUNDO_NEUTRO,
           (True, False): FUNDO_NEUTRO, (True, True): FUNDO_PIORA}
esquema_quadrantes(
    "Como ler a matriz de duração e frequência",
    "Variação do DEC-FI (duração)", "Variação do FEC-FI (frequência)",
    {(False, False): "menos vezes e\nmenos tempo sem energia",
     (False, True): "mais vezes,\nporém menos tempo sem energia",
     (True, False): "menos vezes,\nporém mais tempo sem energia",
     (True, True): "mais vezes e\nmais tempo sem energia"},
    FUNDO_Q)

#### Como ler

Cada ponto é uma distribuidora, e o fundo de cada quadrante tem a cor do esquema. Pontos muito distantes ficam presos à borda, como losangos, com os valores reais ao lado do nome. A estrela é o Brasil. Em azul, as citadas na conclusão que melhoraram nas duas medidas; em vermelho, as citadas que pioraram nas duas.

In [ ]:
x, y = ranking["dec_fi_var_pct"], ranking["fec_fi_var_pct"]
quadrante = pd.Series(list(zip(x >= 0, y >= 0)), index=ranking.index)
DESTAQUE_BOM = ["EQUATORIAL GO", "EQUATORIAL AL"]
DESTAQUE_RUIM = ["ELETROPAULO", "LIGHT SESA", "ENEL RJ", "EMS"]
cores = pd.Series(COR_PONTO, index=ranking.index)
cores[DESTAQUE_BOM] = COR_MELHORA
cores[DESTAQUE_RUIM] = COR_ALERTA

fig, ax = plt.subplots(figsize=(10, 7))
matriz_quadrantes(ax, x, y, cores, DESTAQUE_BOM + DESTAQUE_RUIM, FUNDO_Q)
bx = var_pct(serie.loc[ANO_INI, "dec_fi"], serie.loc[ANO_FIM, "dec_fi"])
by = var_pct(serie.loc[ANO_INI, "fec_fi"], serie.loc[ANO_FIM, "fec_fi"])
ax.scatter(bx, by, marker="*", s=220, color=COR_BRASIL, edgecolor="white", linewidth=0.8, zorder=5)
ax.annotate("Brasil", (bx, by), xytext=(-8, 6), textcoords="offset points", ha="right",
            fontsize=8, fontweight="bold", color=COR_BRASIL, zorder=5)
ax.set_xlabel(f"Variação do DEC-FI (duração), {ANO_INI} a {ANO_FIM} (%)")
ax.set_ylabel(f"Variação do FEC-FI (frequência), {ANO_INI} a {ANO_FIM} (%)")
ax.set_title("Duração e frequência das interrupções, por distribuidora")
plt.tight_layout()
plt.show()

rotulos = {(False, False): "menos vezes e menos tempo", (False, True): "mais vezes, menos tempo",
           (True, False): "menos vezes, mais tempo", (True, True): "mais vezes e mais tempo"}
for q, nome in rotulos.items():
    membros = list(quadrante[quadrante == q].index)
    print(f"{nome:<27}: {len(membros):>2} | {', '.join(membros)}")
print(f"Brasil.....................: DEC-FI {bx:+.1f}%, FEC-FI {by:+.1f}%")

**Conclusão:**

11 distribuidoras reduziram as duas medidas, com as maiores quedas combinadas na EQUATORIAL GO (−39% na duração e −34% na frequência) e na EQUATORIAL AL (−38% e −36%). Outras 11 pioraram nas duas (a CEEE-D, na duração, com a frequência estável), com destaque para ELETROPAULO (+129% e +48%), LIGHT SESA (+48% e +41%), EMS (+32% e +40%) e ENEL RJ (+17% e +45%). Em 7, as interrupções ficaram menos frequentes, porém mais longas; em 4, mais frequentes, porém mais curtas. O Brasil fica perto da origem (+5,4% e +1,1%): a média esconde dois grupos de mesmo tamanho andando em sentidos opostos.

## A regra de agregação do PRODIST muda o resultado?

O PRODIST define como juntar os meses em um indicador anual: cada mês pesa pelo número de consumidores daquele mês, e o total é dividido pela média mensal de consumidores. No mercado, é comum simplificar somando os doze meses. As duas contas só coincidem quando o número de consumidores não muda ao longo do ano. Este trabalho usa a regra e guarda a soma simples ao lado, para medir a diferença.

#### Como ler

As barras mostram as dez maiores diferenças entre a regra e a soma simples, entre todas as apurações anuais. Ao lado, a diferença em horas e o quanto ela representa do DEC-FI daquele ano.

In [ ]:
dif = anual.assign(abs_dif=anual["dif_vs_soma_simples"].abs(),
                   rel=anual["dif_vs_soma_simples"].abs() / anual["dec_fi"] * 100)
maiores = dif.sort_values("abs_dif", ascending=False).head(10).iloc[::-1]
rotulos_y = [f"{s} {a}" for s, a in zip(maiores["sig_agente"], maiores["ano"])]

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.barh(rotulos_y, maiores["abs_dif"], color=COR_NEUTRA, height=0.7)
for i, (v, r) in enumerate(zip(maiores["abs_dif"], maiores["rel"])):
    ax.text(v + 0.005, i, f"{br(v)} h ({br(r, 1)}% do DEC-FI)", va="center", fontsize=7.5, color=COR_TEXTO)
ax.set_xlim(0, maiores["abs_dif"].max() * 1.6)
ax.set_xticks([])
ax.spines["bottom"].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.set_xlabel("Diferença entre a regra do PRODIST e a soma simples dos meses, em horas")
ax.set_title("As dez maiores diferenças entre a regra e a simplificação")
plt.tight_layout()
plt.show()

print(f"apuracoes anuais............: {len(dif)}")
print(f"com diferenca ate 0,05 h....: {(dif['abs_dif'] <= 0.05).sum()}")
print(f"maior diferenca.............: {dif['abs_dif'].max():.2f} h")

**Conclusão:**

Em 120 das 132 apurações anuais (33 distribuidoras em quatro anos), a diferença entre a regra do PRODIST e a soma simples é de até 0,05 hora. A maior é de 0,44 hora, na RGE SUL em 2024, 1,2% do DEC-FI daquele ano. Nesta base, a simplificação é pequena diante das variações analisadas, que são de horas. Mantive a regra por ser a norma, não por ela mudar o resultado.

## Conclusão

**A continuidade melhorou?** Pela régua regulatória, sim: o DEC e o FEC normativos do conjunto caíram 16,6% e 14,6% entre 2022 e 2025. Pela régua deste trabalho, que conta também os dias críticos e as emergências, não: o DEC-FI subiu 5,4%, e o FEC-FI ficou estável.

- **A média esconde trajetórias opostas.** 11 distribuidoras reduziram a duração e a frequência das interrupções; outras 11 pioraram nas duas.
- **Quem partia de patamar alto reduziu mais.** As três maiores reduções em horas são do grupo Equatorial (AL, GO e MA), todas acima de 28 horas em 2022.
- **Os anos do meio foram os piores para a maioria.** A comparação entre as pontas depende dos anos escolhidos, e o ranking deve ser lido junto com a trajetória.
- **A regra de agregação não muda a leitura nesta base.** A diferença para a soma simples não passa de 0,44 hora.

A comparação entre continuidade e reclamações de falta de energia está em `complaints/04_analysis`.

## Pendências documentadas

| Item | Situação | O que falta |
|---|---|---|
| Eventos climáticos | Picos de 2023 e 2024 identificados, causa não apurada | Estudo em notebook próprio, separando os meses de evento, após a conclusão do trabalho |
| Validação externa | Não feita | Confrontar o ranking com o DGC (Desempenho Global de Continuidade) publicado pela ANEEL, item 818 do PRODIST Módulo 8 |
| Parcelas do DEC-FI | Diferença para o normativo explicada pela definição | Abrir o DEC-FI por parcela (interrupções comuns, dias críticos, emergências) para medir o peso de cada uma |

## Autoavaliação desta etapa

**O que esta etapa entregou.** Respondi a quatro perguntas sobre a continuidade com cinco gráficos, sempre com o Brasil ou o indicador normativo como referência. O principal achado: pela régua regulatória a continuidade melhorou entre 2022 e 2025; pela régua do consumidor, que inclui dias críticos e emergências, não.

**O que mudou no caminho.** A primeira versão da análise de continuidade era uma tabela e um gráfico de barras dentro da Gold. Na revisão final, separei a análise e acrescentei a trajetória ano a ano, a matriz de duração e frequência e o confronto com o indicador normativo. A explicação sobre os períodos de comparação entrou nos dois notebooks de análise, porque não há ordem garantida de leitura.

**O que ficou em aberto.** A causa dos picos de 2023 e 2024, o peso de cada parcela no DEC-FI e a validação externa do ranking.

**O que eu faria diferente.** Definiria desde o início um período de comparação comum às duas fontes, para não haver duas variações do Brasil que o leitor precisa conciliar.
